<a href="https://colab.research.google.com/github/tanjun8802/Mase_EDGE/blob/jtan%2Fdev/EDGE_NAS_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## EDGE — Optuna study and ExecuTorch (XNNPACK PT2E) export

Optuna searches **per-layer XNNPACK-valid static INT8** (symmetric weights, asymmetric **static** activations; per-channel or per-tensor) plus **no quantization** per `Conv2d` / `Linear`, and **L1 unstructured pruning** via `torch.nn.utils.prune`. **Dynamic** activation PT2E is omitted here because it inserts `quantized_decomposed` ops that commonly fail ExecuTorch `to_executorch()` (missing out variants) on many torch+executorch versions. Export uses `prepare_pt2e` / `convert_pt2e` and `to_edge_transform_and_lower` with `XnnpackPartitioner`. Phone metrics still come from ADB + the Image Classification demo app.

In [ ]:
# # For colab use

# # !git clone --branch jtan/dev https://github.com/tanjun8802/Mase_EDGE.git 
# # %cd /content
# # !rm -rf Mase_EDGE/mase
# # %cd Mase_EDGE
# !git submodule update --init --recursive
# %cd mase/
# !pip install -e .

In [ ]:
# !git submodule update --init --recursive
# %cd ./mase
# %pip install -e ".[executorch]"

In [ ]:
# # Get the dataset (Tiny ImageNet)

# %cd /content/Mase_EDGE
# !wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
# !unzip tiny-imagenet-200.zip

In [ ]:
# pip install executorch  (and compatible torch / torchao per ExecuTorch docs)
import optuna
import torch
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd
for c in (_cwd, _cwd.parent, _cwd.parent.parent):
    if (c / "edge_study").is_dir():
        PROJECT_ROOT = c
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from EDGE_device.device_specifications import (
    get_adb_output,
    get_adb_sh_c,
    get_hardware_for_phone,
    load_phone_specs,
)

from edge_study import settings
from edge_study import (
    QUANT_LAYER_NAMES,
    XNNPACK_LAYER_QUANT_CHOICES,
    optuna_param_name_for_module,
    edge_optuna_config,
    load_model_and_train,
    edge_optimise_model,
    edge_export_model,
    edge_host_val_sanity_check,
    edge_benchmark_trial,
    edge_poll_metrics,
    clear_edge_metrics_json_on_device,
    move_dataset_to_app,
    move_pte_to_app,
    make_objective,
)


In [ ]:
settings.init_default_dataset_path()

IMAGE_CLASSIFICATION_APP_ID = settings.IMAGE_CLASSIFICATION_APP_ID
EDGE_DEVICE_DATASET_PATH = settings.EDGE_DEVICE_DATASET_PATH
EDGE_EVAL_SPLIT = settings.EDGE_EVAL_SPLIT
EDGE_METRICS_POLL_TIMEOUT_FLAG = settings.EDGE_METRICS_POLL_TIMEOUT_FLAG
EDGE_BENCHMARK_MAX_IMAGES_FLAG = settings.EDGE_BENCHMARK_MAX_IMAGES_FLAG
BASE_STATE_DICT = settings.BASE_STATE_DICT
RESNET18_PT_PATH = settings.RESNET18_PT_PATH
MOBILENET_V3_PATH = settings.MOBILENET_V3_PATH
EPOCHS = settings.EPOCHS
LR = settings.LR
BATCH_SIZE = settings.BATCH_SIZE
PROJECT_ROOT = settings.PROJECT_ROOT

if EDGE_DEVICE_DATASET_PATH:
    print(f"EDGE_DEVICE_DATASET_PATH → {EDGE_DEVICE_DATASET_PATH}")
else:
    print(
        "WARN: No dataset path (mase/tiny-imagenet-200 or tiny-imagenet-200 missing). "
        "Trial 0 will not push data; expect acc=0 / lat=999 from the app."
    )


### 1. Load Metrics from Android Phone

If you are trying to optimise and deploy the model on an Android phone, please plug in the phone to the laptop and make sure the developer mode is on. In settings make sure debugging mode is turned on. Run the following code cell to extract the hardware information. A JSON file will be created to hold the information and it will be used in the configuration setup.

In [ ]:
hw = get_hardware_for_phone() # Make sure the phone is connected and ADB is set up
print("Optimizing for:", hw["model"])
print(hw["gpu"], hw["cpu_cores"], hw["ram_gb"])

In [ ]:
# # NPU/NNAPI hardware check (non-reference devices indicate accelerators like NPU)
# # Reload so notebook picks up latest ``get_adb_sh_c`` (e.g. ``timeout_sec``) without kernel restart.
# import importlib
# import EDGE_device.device_specifications as _adb_spec_mod
# importlib.reload(_adb_spec_mod)
# get_adb_output = _adb_spec_mod.get_adb_output
# get_adb_sh_c = _adb_spec_mod.get_adb_sh_c

# # ``get_adb_output`` / ``get_adb_sh_c`` use a timeout (default 20s, env ``ADB_SHELL_TIMEOUT_SEC``).
# # Avoid ``service list`` here — it often blocks for a long time on real devices.
# checks_simple = {
#     "ro.hardware": "shell getprop ro.hardware",
# }
# checks_shell = {
#     # ``grep -i npu`` false-positives on ``c2inputsurface`` (substring ``input`` → n-p-u)
#     "npu_prop": "getprop | grep -i npu | grep -vi c2inputsurface || true",
#     "cpuinfo": "head -20 /proc/cpuinfo",
#     "nn_hal": "ls /vendor/lib/hw/ /vendor/lib64/hw/ 2>/dev/null | grep neuralnetworks || echo none",
#     "init_neural_svcs": "getprop | grep -i init.svc.neural || true",
# }

# results = {k: get_adb_output(v) for k, v in checks_simple.items()}
# for k, script in checks_shell.items():
#     results[k] = get_adb_sh_c(script, timeout_sec=15.0)

# # Heuristic NPU detection
# npu_indicators = ["npu", "hexagon", "qcom", "kirin", "mali", "neuralnetworks"]
# has_npu = any(any(ind in res.lower() for ind in npu_indicators) for res in results.values())
# print(f"NPU/Accel likely: {'YES' if has_npu else 'NO'}")
# print("Full results:", results)

In [ ]:
# specs = load_phone_specs('EDGE_device/device_specifications.json')          # uses default PHONE_SPECS_FILE
# print("Specs dict:", specs)
# print("Keys:", list(specs.keys()))

# if specs:
#     hw = list(specs.values())[0]
#     print(hw["model"], hw["gpu"], hw["cpu_cores"], hw["ram_gb"])
# else:
#     print("No specs found; JSON file missing or empty.")

### 2. Defining the Objective Function

In [ ]:
objective = make_objective()


### 3. Optuna Study

In [ ]:
# Create + run study
# If you shrink XNNPACK_LAYER_QUANT_CHOICES, SQLite-backed studies may error on suggest_categorical;
# use a new study_name (e.g. mv3_edge_opt_v2) or delete optuna_mv3.db.
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="mv3_edge_opt",
    storage="sqlite:///optuna_mv3.db",  # Save progress
    load_if_exists=True,
)

# Run 50 trials
study.optimize(objective, n_trials=20)

print(f"\nBest trial #{study.best_trial.number}")
print(f"   Score: {study.best_value:.3f}")
print("   Config:", study.best_params)

# Quick viz

optuna.visualization.plot_optimization_history(study).show()


In [ ]:
optuna.visualization.plot_optimization_history(study).show()

### Extract the best model

In [ ]:
# Get best config (user_attrs include full layer_quant dict from the trial)
best_trial = study.best_trial
best_config = {
    k: v
    for k, v in best_trial.user_attrs.items()
    if k
    in (
        "prune_ratio",
        "backend",
        "use_mixed_delegation",
        "delegation_plan",
        "layer_quant",
    )
}
# Fallback if attrs missing (e.g. old study): rebuild layer_quant from params
if "layer_quant" not in best_config and best_trial.params:
    best_config["prune_ratio"] = best_trial.params.get(
        "prune_ratio", best_config.get("prune_ratio", 0.0)
    )
    best_config["backend"] = "xnnpack"
    best_config["use_mixed_delegation"] = False
    best_config["delegation_plan"] = None
    best_config["layer_quant"] = {
        mod: best_trial.params[optuna_param_name_for_module(mod)]
        for mod in QUANT_LAYER_NAMES
    }

best_model = edge_optimise_model(best_config, enable_qat=False)
final_pte = edge_export_model(best_model, 999, best_config)

print(f"Best model: {final_pte}")
print("Deploy with: adb push", final_pte, "/sdcard/mobilenetv3_best.pte")



In [ ]:
edge_benchmark_trial(None, "pte_files/mobilenetv3_trial_999.pte", None, "val", eval_only=True)
